<div style="background: linear-gradient(135deg, #e9f2fb 0%, #f7fbff 100%); padding: 18px 20px; border: 1px solid #d5e3f0; border-radius: 10px; text-align: center;">
  <div style="font-size: 32px; font-weight: 700; color: #1f3b57; letter-spacing: 0.2px;">NX-414 Brain-like Computation and Intelligence</div>
</div>


<div style="text-align: center; font-size: 21px; font-weight: 600; color: #36536b; margin-top: 6px;">Project Notebook — Spring 2026</div>


<div style="text-align: center; color: #4f6478; font-size: 16px; margin-bottom: 10px;">Brain–Model Alignment Across Neural Recording Modalities</div>
<div style="text-align: center; color: #6b7280; font-size: 13px;">Prepared by: Abdulkadir Gokce</div>

---


# Group Information

Fill in this section at the top of your notebook and report.

- **Group member 1:** Martina Baroffio, 406109, martina.baroffio@epfl.ch  
- **Group member 2:** Clotilde Cerruti, 413476, clotilde.cerruti@epfl.ch  
- **Group member 3 (if applicable):** Martina Lodetti, 405916, martina.lodetti@epfl.ch  
MODIFICA FUFFA
---

# What You Must Submit

Submit the following files:

1. **One Jupyter notebook** containing your full analysis.
2. **Any supporting Python scripts** needed to run the notebook.
3. **Figures that are part of your notebook answers** should be embedded and rendered in notebook Markdown.
4. **One PDF report** of **up to 2 pages**, **excluding references**, with **no appendix**.
5. **One zip archive** named exactly as:

```text
nx414_{SCIPER1}_{SCIPER2}_{SCIPER3}.zip
```

If your group has fewer than three members, reduce the number of `_SCIPER` fields accordingly.

## Submission Rules

- **Clear all notebook outputs before submission.**
- If outputs are not cleared, we will clear them ourselves and grade the cleaned notebook.
- Submit only the code required to reproduce your results.
- **Do not submit model weights.**
- **Do not submit CSV files or other large derived result dumps.**
- Keep the archive lightweight and reproducible.
- For the **final notebook**, any figure you want to present as part of your scientific argument should be **embedded in Markdown with accompanying interpretation**, rather than left as a raw cell output with no explanation.

Failure to follow these instructions may reduce your final grade.

## Use of LLMs

You may use LLM-based tools to help you write code, debug, or improve explanations. However, you remain fully responsible for the **correctness**, **quality**, and **clarity** of everything you submit, including both the notebook and the report.

In particular:

- check that any generated code actually runs and does what you claim it does,
- verify that any scientific statement or interpretation is correct,
- make sure the final writing sounds like a clear academic report written for this course,
- avoid vague, overly polished, or context-free text,
- avoid fancy wording or unnecessarily complex sentences that do not add clarity.

If you use an LLM, revise the output so that your submission reads naturally, is specific to your actual results, and does not look like generic generated text.

Failure to do so may result in **a point deduction**.

## Expected scope

Because this project spans roughly **three weeks** and counts for **30% of the final course grade**, the expected output is closer to a **compact course project** than to a one-week homework notebook. Your submission should therefore read like a small empirical study: it should be clearly structured, contain short written interpretations throughout, compare alternatives systematically, and end with a coherent synthesis of your main findings.

A strong notebook will not only run end-to-end, but will also explain **why** each analysis is being performed, what each metric is meant to capture, and what the results imply about the strengths and limitations of the models and datasets.

At the same time, **some parts of the project are intentionally left a bit loose**. This is by design: beyond implementing the required core analyses, you are expected to make reasonable scientific choices, justify them clearly, and show some ingenuity in how you explore the data and compare models.

## Suggested 3-week pacing

Use the notebook structure below to organize your work over the three weeks.

- **Week 1:** complete **Section 0** and **Section 1**. Understand the datasets, verify stimulus matching, inspect the processed responses, and start the visualization and reliability analyses.
- **Week 2:** complete the required analyses in **Section 2**. In this section, you must complete both the representational and predictive parts of the project. Begin **Section 3** by brainstorming possible extensions and sketching out a plan for your chosen extension.
- **Week 3:** complete **Section 3**, polish the notebook, select the strongest figures, and write the 2-page report.


---

# 0. Introduction and Setup

## 0.1 Project goal

In this project, you will study how neural responses from different recording modalities align with features extracted from two vision models. More specifically, you will work in the standard **brain–model alignment** setting: a model processes an image, a candidate internal layer is selected, and that representation is compared to measured neural responses using representational and predictive metrics.

The notebook is organized around four sections:

- **Section 0:** introduction, setup, and understanding the provided resources.
- **Section 1:** dataset inspection, visualization, and noise ceiling estimation.
- **Section 2:** brain–model alignment through both representational metrics and predictive linear models.
- **Section 3:** an open-ended extension beyond the baseline pipeline.

## 0.2 Why task-optimized models?

Task-optimized neural networks are among the most useful current **in-silico models of sensory cortex**. The central idea is simple: instead of hand-designing a model to mimic every biological detail, we optimize a model to perform a meaningful visual task and then ask whether its internal representations resemble those found in the brain. This approach has been highly influential because models trained to solve vision tasks often develop representations that predict activity along the visual hierarchy surprisingly well.

These models are useful scientifically because they provide **testable computational hypotheses**. If a model layer predicts neural responses well, that does not mean the brain literally implements the same mechanism, but it does suggest that the layer may encode information in a similar format or at a similar level of abstraction. Brain–model alignment is therefore a way to ask not just whether a model is accurate on a task, but whether it organizes visual information in a brain-relevant way.

## 0.3 Why compare multiple modalities?

A single recording modality gives only a partial view of neural computation. In this project, you will work with **electrophysiology, EEG, and fMRI**, which differ in temporal resolution, spatial resolution, and what exactly is measured. Looking across modalities helps you see which conclusions are robust and which depend on the measurement scale.

## 0.4 Learning goals

By the end of this project, you should be able to:

- inspect and summarize neural datasets from multiple modalities,
- visualize neural signals and data quality,
- implement and compare **two noise ceiling estimators**,
- implement **RSA** and **unbiased linear CKA**,
- fit **linear encoding models** from model features to neural responses,
- compare alignment across **models, layers, ROIs, and metrics**,
- interpret what each alignment metric captures,
- design and evaluate one meaningful extension beyond the baseline pipeline.

A strong submission should therefore demonstrate both **technical correctness** and **scientific reasoning**: beyond obtaining scores, you should be able to explain why a dataset is noisy, why one layer may outperform another, and why representational and predictive metrics sometimes disagree.

## 0.5 Provided data

All main data are stored in `/shared/NX-414/data/`.

### Background: processed data derivatives

The files provided for this project are **not raw neural recordings**. They are already processed, analysis-ready derivatives produced with modality-appropriate pipelines. This is important scientifically: many of your results will depend not only on the model features, but also on preprocessing choices such as repetition averaging, denoising, response-window selection, voxel/channel filtering, and how reliability is estimated.

We performed the preprocessing for you because these pipelines often require substantial modality-specific expertise, time, and compute.

At a high level, the datasets used here were prepared as follows:

- **TVSD (macaque electrophysiology)** — normalized multi-unit responses from ventral-stream areas. Responses were z-scored within session, firing rates were averaged in an analysis window centered on each site’s response peak, low-reliability channels were excluded, and repeated test responses were averaged for evaluation.
- **THINGS-EEG2 (human EEG)** — source EEG responses resampled to **100 Hz**. Noise ceilings were computed per subject, channel, and time point, and repetitions were averaged within train and test splits.
- **NSD (human fMRI)** — **b3 single-trial beta estimates** in `func1pt8mm` space, derived using voxel-wise HRF fitting, GLMdenoise, and ridge regression. Analyses are restricted to ROI-defined visually responsive voxels, low-reliability voxels are filtered out, and responses are averaged across available repetitions.

You are **not** expected to re-run the full preprocessing pipelines. You **are** expected to understand what kinds of neural quantities you are analyzing, what has already been averaged or denoised, and how these choices affect interpretation.

### Main neural datasets

- **`tvsd.h5`** — macaque electrophysiology from **2 monkeys**, with **22,248 train** and **100 test** stimuli, covering **V1, V4, and IT**.
- **`things_eeg2.h5`** — human EEG from **10 subjects**, with **16,540 train** and **200 test** stimuli, with region groupings such as **occipital**, **parietal**, **temporal**, **frontal**, **central**, **occipital_parietal**, and **whole_brain**.
- **`nsd_func1pt8mm_individualROIs.h5`** — human fMRI from **8 subjects**, with roughly **9,000 train** and **1,000 test** stimuli per subject, across multiple visual ROIs.

### Additional files

- `things_eeg2-test_reps.h5`  
  EEG test responses **with repetitions and without averaging**.  
  Use this file to implement and compare **two noise ceiling estimators**.

- `nsd-subj01-ncsnr-{lh,rh}.mgh`  
  Surface-based NSD reliability values for `subj-01` on **fsaverage**.  
  Use these to visualize cortical reliability and convert **ncsnr** into **noise ceiling**.

### Neural response shapes

- **TVSD:** `(n_stimuli, n_units)`
- **EEG2:** `(n_stimuli, n_channels, n_timepoints)`
- **NSD:** `(n_stimuli, n_voxels)`

For EEG, the time axis contains **80 time points** sampled at **100 Hz**, covering **0.0 s to 0.8 s**.

### Noise ceilings

Noise ceilings are stored per target:

- per neuron for **TVSD**,
- per channel × time point for **EEG2**,
- per voxel for **NSD**.

They are stored as **percent reliability**.  
To convert them to the range `[0, 1]`, divide by `100`.

In this project, the provided noise ceilings are mainly intended for **predictive metrics** such as **Pearson correlation** and **explained variance**. They reflect the reliability of the neural responses and therefore define an upper bound on how well any model can predict those responses.

When you compute predictive metrics, you should apply a noise ceiling correction to account for this upper bound. The standard idea is simple: divide the raw predictive score by the corresponding noise ceiling value for that target.

The provided noise ceilings are defined for **explained variance**. If you want to apply the same logic to **Pearson correlation**, first convert the explained-variance ceiling into a correlation ceiling by taking the element-wise square root, and then divide the raw correlation by that quantity. For a more detailed discussion of noise ceiling correction, see van Bree et al. (2025).

You may therefore report both **raw** and **noise-ceiling-corrected** predictive scores where appropriate.


For example, if the provided ceiling is an explained-variance reliability estimate, you can compute a noise-corrected Pearson correlation as:

```python
r_nc = r / np.sqrt(ev_ceiling)
```

where `r` is the raw Pearson correlation and `ev_ceiling` is the explained-variance ceiling expressed on the range `[0, 1]`.

By contrast, **RSA** and **CKA** should typically be reported as **raw values** in this project. Noise ceilings for representational similarity metrics require a different methodology and are **not** provided here.


Do **not** apply this correction directly to **RSA** or **CKA**.

### Stimulus identifiers

- **TVSD / EEG2:** byte strings such as `b'aardvark/aardvark_01b.jpg'`
- **NSD:** integer stimulus IDs

## 0.6 Model features

All extracted features are stored in `/shared/NX-414/extracted_features/`.

The feature files contain **internal activations extracted from multiple candidate layers** while the models process the same images shown in the neural experiments. You can think of each layer as a representation matrix of shape roughly **stimuli × features**. These layer-wise representations are what you will compare to the brain using RSA, CKA, and encoding models.

Feature extractions across models were made tractable by projecting activations to **30,000 dimensions** using a random projection. The provided feature files follow that same idea. In practice, this means you can treat the feature vectors as compact surrogates for the original activations while still performing meaningful alignment analyses.

### Model A: `adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0`

- Architecture: **ResNet-152**
- Pretraining: ImageNet + adversarial fine-tuning
- Available feature files:
  - `things_stimuli.h5`
  - `nsd_stimuli.h5`

### Model B: `Qwen3-VL-2B-Instruct`

- Architecture: **vision-language transformer**
- Available feature files:
  - `things_stimuli.h5`
  - `nsd_stimuli.h5`

For both models, layers have been projected to **30,000 dimensions** using random projections.

### Feature extraction note

For this project, feature extraction has already been done for you. Your job is therefore not to run the vision models on raw images, but to understand **which layer** to use, how to match feature rows to neural stimuli, and what different layers reveal about representational hierarchy.

### Important note for NSD

For NSD, the feature files contain features for **all 73,000 images**, but each subject saw only a subset (~9,000).  
You must therefore select feature rows using the subject-specific NSD stimulus IDs.

## 0.7 Matching features to neural responses

You must match neural responses and features through the stimulus IDs.

### THINGS-based datasets: TVSD and EEG2

For these datasets, both neural IDs and feature IDs are byte strings. Matching should therefore be exact.

### NSD

For NSD, both sides use integer IDs. The feature file contains all 73,000 stimuli, while each subject has only a subset. Use the subject-specific NSD stimulus IDs to select the corresponding feature rows.

### Recommended procedure

```python
feat_ids = feature_file['ids'][:]
id_to_feat_idx = {id_: i for i, id_ in enumerate(feat_ids)}
feat_idx = np.array([id_to_feat_idx[x] for x in neural_ids])
```

To make HDF5 reads efficient:

1. build the index array,
2. sort indices before reading,
3. load only the required rows,
4. restore the original order.

## 0.8 General analysis rules

### Train/test discipline

Do **not** use the test split for model selection or hyperparameter tuning. Use a validation split or cross-validation within the training data.

### EEG targets

For EEG, you may either:

- flatten the targets to `(n_stimuli, n_channels * n_timepoints)`, or
- fit separate models per channel × time point.

Either choice is acceptable, but you must explain your decision clearly.

### Predictive metrics

Report the following predictive metrics where appropriate:

- `pearsonr`
- `pearsonr_nc`
- `explained_variance`
- `explained_variance_nc`

The provided EEG signals are not filtered like the other datasets. As a result, some low-reliability channels or time points can produce unstable predictive scores. When averaging predictive metrics over EEG targets, apply an on-the-fly filter such as **noise ceiling < 0.1**.

### Representational metrics

Also report:

- `RSA`
- `CKA`
- `encoding-RSA`
- `encoding-CKA`

Encoding RSA/CKA is a hybrid metric where you compute RSA or CKA between the predicted neural responses (from the linear encoding model) and the actual neural responses, instead of just comparing model activation directly. This can help you understand whether the linear encoding model captures the representational geometry of the neural data, beyond just predicting individual response values. You can refer to Conwell et al. (2024) for more details on this approach. 

Use only the test split for computing representational metrics.

Do **not** noise-correct RSA or CKA using the predictive-metric procedure.

## 0.9 Setup and data loading

Your notebook should begin with a short introduction, clear imports, utility functions, and a brief verification that the provided files are correctly organized and matched.

**Section 0 is required but not graded separately.** It is treated as setup for the rest of the project. Missing or incorrect setup may reduce scores in later sections if it affects correctness or reproducibility.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Import the required packages. 
- Define utility functions.
- Load metadata for all datasets.
- Inspect the structure of each `.h5` file.
- Load feature metadata for both models.
- Verify that feature IDs and neural stimulus IDs match.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Dataset and feature overview:** one compact table or printed summary covering all neural datasets and both feature sets.
2. **Stimulus-matching verification:** explicit checks or assertions showing that stimulus matching works for THINGS-based datasets and for NSD.
3. **Short structural summary:** a short written note describing the main structural differences across datasets.

**1. Dataset and Feature Overview**

| Dataset | Top-level groups | Subjects | Split | Neural data path | Stimulus IDs path |
|---------|-----------------|----------|-------|-----------------|-------------------|
| TVSD | `noise_ceilings`, `train`, `test` | `monkeyF`, `monkeyN` | `train`, `test` | `{split}/neural_data/{monkey}/{ROI}` | `{split}/stimulus_ids` |
| EEG2 | `noise_ceilings`, `noise_ceilings_train`, `train`, `test` | `sub-01` → `sub-10` | `train`, `test` | `{split}/neural_data/{sub}/{region}` | `{split}/stimulus_ids` |
| NSD | `noise_ceiling_full`, `noise_ceilings`, `roi_labels`, `roi_labels_nc`, `train`, `test` | `subj01` → `subj08` | `train`, `test` | `{split}/neural_data/{subj}/{ROI}` | `{split}/stimulus_ids/{subj}` |

? Is this too compact and we need to show better the structure with also dimensions here
ADD FEATURE TABLE

**2. Stimulus-matching verification**
Stimulus matching is handled by the utility function match_feature_idx, which aligns feature and neural stimulus IDs. Assertions are included to verify correct one-to-one correspondence between feature and neural stimuli for both THINGS-based datasets and NSD.

**3. Short structural summary**
TVSD contains spiking responses from two monkeys across three visual areas (IT, V1, V4), with no time axis. EEG2 contains EEG responses from 10 human subjects with a full time axis (80 timepoints) across 7 scalp regions. NSD contains fMRI BOLD responses from 8 human subjects across 28 cortical ROIs, with subject-specific voxel counts and MRI volumes. The key structural differences are: TVSD has no subject axis (only monkeys × ROIs), EEG2 adds a time dimension, and NSD has the richest ROI coverage but no time axis.



<div style="background:#eef8f4; border-left:4px solid #5b9a7a; padding:8px 12px; border-radius:6px; font-weight:700; color:#285943;">Questions you should answer</div>

- How many stimuli are available in each dataset?
- What is the shape of the neural response tensor in each dataset?
- Which datasets contain subjects, ROIs, repetitions, channels, or time points?
- What are the feature dimensionalities across layers?

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 0</strong><br>Summarize the structure of the three datasets and the two feature sets in 4–6 sentences. Clearly state which dataset is most complex structurally and why.</div>

The three datasets differ in structure and complexity. In TVSD, the number of stimuli is 22248 in the training set and 100 in the test set, with neural responses organized as image presentations × neurons, where the second dimension varies depending on the region (V1, V4, IT). EEG2 contains 16540 training stimuli and 200 test stimuli, with data structured across stimuli, channels, and 80 time points per trial. NSD is more complex, as the number of stimuli varies across subjects (e.g., 9000, 8481, 8302 for training and 1000, 930, 907 for testing) and includes multiple ROIs defined in 3D MRI space. While TVSD and EEG2 have relatively fixed structures, NSD includes subject-specific data, making it the most structurally complex dataset. Finally, feature representations from both models are organized as stimuli × features and projected to a fixed dimensionality (30000) across layers; the feature sets cover all stimuli in both NSD (73000 stimuli) and THINGS-based datasets (26107 stimuli), ensuring consistent comparison across models.


In [ ]:
# TODO: imports
# TODO: define paths
# TODO: inspect dataset structure
# TODO: verify stimulus matching

**Import libraries and packages**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn import metrics
from scipy import stats
import nibabel as nib
from nilearn import plotting as nlplt
import h5py
import torch

**Define utility functions**

In [ ]:
def explore_h5(path, indent=0):
    """Explore h5 file in detail."""
    with h5py.File(path, 'r') as f:
        def print_structure(name, obj):
            print(' ' * indent + name, ':', type(obj).__name__, end='')
            if isinstance(obj, h5py.Dataset):
                print(f'  shape={obj.shape}, dtype={obj.dtype}')
            else:
                print()
        f.visititems(print_structure)

def load_h5_attrs(path):
    """
    Recursively collect all HDF5 attributes from every group and dataset.
    Returns a nested dict: {group_or_dataset_path: {attr_name: attr_value, ...}, ...}
    Root-level attributes are stored under the key '/'.
    """
    attrs = {}
    with h5py.File(path, 'r') as f:
        if f.attrs:
            attrs['/'] = dict(f.attrs)
        def _collect(name, obj):
            if obj.attrs:
                attrs[name] = dict(obj.attrs)
        f.visititems(_collect)
    return attrs

def load_neural_IDs(path, split='train', subjects=None):
    """    Load stimulus IDs for a given split (train or test).
    Handles both single (TVSD, EEG2) and per-subject (NSD) structures.
    """
    with h5py.File(path, 'r') as f:
        ids_key = f'{split}/stimulus_ids'
        obj = f[ids_key]
        
        if isinstance(obj, h5py.Dataset):
            # TVSD / EEG2: single stimulus_ids array
            return obj[:]   

        elif isinstance(obj, h5py.Group):
            # NSD: one array per subject
            if subjects is None:
                subjects = list(obj.keys())
            return {sub: obj[sub][:] for sub in subjects}

def load_feature_IDs(path):
    """Load stimulus IDs from a feature h5 file."""
    with h5py.File(path, 'r') as f:
        return f['ids'][:]

def match_feature_idx(feature_file, neural_ids):
    """
    Build feature indices aligned to neural_ids,
    following efficient 5 read procedure.
    """
    # Step 1: build index array

    # load all stimulus IDs from feature file
    with h5py.File(feature_file, 'r') as f:
        feat_ids = f['ids'][:]
        # Builds a dictionary: {id: index} for fast lookup
        id_to_feat_idx = {id_: i for i, id_ in enumerate(feat_ids)} 
        # Ror each neural stimulus ID, find its position in the feature stimulu ID array
        feat_idx       = np.array([id_to_feat_idx[x] for x in neural_ids]) 
        
        # Step 2: sort indices for efficient HDF5 reads
        sort_order     = np.argsort(feat_idx)
        feat_idx_sorted = feat_idx[sort_order]
    
        # Step 3: load only required rows (sorted)
        features_sorted = f['ids'][feat_idx_sorted]
        
    # Step 4: restore original order --> = matching the neural ids
    restore_order = np.empty_like(sort_order)
    restore_order[sort_order] = np.arange(len(sort_order))
    matched_ids = features_sorted[restore_order]
    
    return matched_ids

In [ ]:
# Define data paths
TVSD_path = "/shared/NX-414/data/tvsd.h5"
EEG2_path = "/shared/NX-414/data/things_eeg2.h5"
NSD_path = "/shared/NX-414/data/nsd_func1pt8mm_individualROIs.h5"

# Define feature paths
RN_NSD_path = "/shared/NX-414/extracted_features/adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0/nsd_stimuli.h5"
RN_EEG_path = "/shared/NX-414/extracted_features/adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0/things_stimuli.h5"
T_NSD_path = "/shared/NX-414/extracted_features/Qwen3-VL-2B-Instruct/nsd_stimuli.h5"
T_EEG_path = "/shared/NX-414/extracted_features/Qwen3-VL-2B-Instruct/things_stimuli.h5"

**Load dataset metadata**

In [ ]:
attrs = load_h5_attrs(TVSD_path)
print("TVSD attributes:")
for key, value in attrs.get('/', {}).items():
    print(f"  {key}: {value}")  

attrs = load_h5_attrs(EEG2_path)
print("\nEEG2 attributes:")
for key, value in attrs.get('/', {}).items():
    print(f"  {key}: {value}")  

attrs = load_h5_attrs(NSD_path)
print("\nNSD attributes:")
for key, value in attrs.get('/', {}).items():
    print(f"  {key}: {value}")  

**Inspect .5 files**

In [ ]:
#explore_h5(TVSD_path) #--> long output, uncomment to inspect the structure
explore_h5(EEG2_path) #--> long output, uncomment to inspect the structure
#explore_h5(NSD_path) #--> long output, uncomment to inspect the structure


#explore_h5(RN_NSD_path)
#explore_h5(RN_EEG_path)
#explore_h5(T_NSD_path)
#explore_h5(T_EEG_path)

**Load feature metadata**

In [ ]:
attrs = load_h5_attrs(RN_NSD_path)
print("ResNet - NSD attributes:")
for key, value in attrs.get('/', {}).items():
    print(f"  {key}: {value}")  

attrs = load_h5_attrs(RN_EEG_path)
print("ResNet - THINGS attributes:")
for key, value in attrs.get('/', {}).items():
    print(f"  {key}: {value}")  

attrs = load_h5_attrs(T_NSD_path)
print("Transformer - NSD attributes:")
for key, value in attrs.get('/', {}).items():
    print(f"  {key}: {value}")  

attrs = load_h5_attrs(T_EEG_path)
print("Transformer - THINGS attributes:")
for key, value in attrs.get('/', {}).items():
    print(f"  {key}: {value}")  

**Verify feature and stimulus IDs match**

In [ ]:
tvsd_train_ids  = load_neural_IDs(TVSD_path, split='train')
eeg2_train_ids  = load_neural_IDs(EEG2_path, split='train')
nsd_train_ids   = load_neural_IDs(NSD_path,  split='train')  # returns dict per subject

tvsd_test_ids  = load_neural_IDs(TVSD_path, split='test')
eeg2_test_ids  = load_neural_IDs(EEG2_path, split='test')
nsd_test_ids   = load_neural_IDs(NSD_path,  split='test')  # returns dict per subject

print("Neural IDs format: ")
print(f"TVSD: {tvsd_train_ids[:2]}")
print(f"EEG2: {eeg2_train_ids[:2]}")
print(f"NSD: {nsd_train_ids['subj01'][:5]}")

In [ ]:
RN_NSD_ids = load_feature_IDs(RN_NSD_path)
RN_EEG_ids = load_feature_IDs(RN_EEG_path)
T_NSD_ids = load_feature_IDs(T_NSD_path)
T_EEG_ids = load_feature_IDs(T_EEG_path)

print(f"Number of stimuli with NSD: {len(RN_NSD_ids)} (ResNet), {len(T_NSD_ids)} (Transformer)")
print(f"Number of stimuli with THINGS (EEG2-TVSD): {len(RN_EEG_ids)} (ResNet), {len(T_EEG_ids)} (Transformer)")

In [ ]:
# THINGS-based datasets
for split, tvsd_ids, eeg2_ids in [
    ('train', tvsd_train_ids, eeg2_train_ids),
    ('test',  tvsd_test_ids,  eeg2_test_ids),
]:
    for label, ids in [('TVSD', tvsd_ids), ('EEG2', eeg2_ids)]:
        for model, path in [('RN', RN_EEG_path), ('T', T_EEG_path)]:
            matched = match_feature_idx(path, ids)
            assert np.all(matched == ids), f"{label} {model} {split} IDs do not match!"
    print(f"✓ THINGS [{split}]: TVSD ({len(tvsd_ids)}) & EEG2 ({len(eeg2_ids)}) IDs matched for RN & T!")

# NSD dataset
subjects = [f"subj{i:02d}" for i in range(1, 9)]
nsd_matched_RN_ids = {'train': {}, 'test': {}}
nsd_matched_T_ids  = {'train': {}, 'test': {}}

for split, nsd_ids in [('train', nsd_train_ids), ('test', nsd_test_ids)]:
    for sub in subjects:
        neural_ids = nsd_ids[sub]
        nsd_matched_RN_ids[split][sub] = match_feature_idx(RN_NSD_path, neural_ids)
        assert np.all(nsd_matched_RN_ids[split][sub] == neural_ids), f"RN IDs do not match for {sub} [{split}]!"
        nsd_matched_T_ids[split][sub] = match_feature_idx(T_NSD_path, neural_ids)
        assert np.all(nsd_matched_T_ids[split][sub] == neural_ids), f"T IDs do not match for {sub} [{split}]!"
    print(f"✓ NSD [{split}]: all {len(subjects)} subjects matched for RN & T!")

---

# 1. Inspection, Visualization, and Noise Ceiling Estimates

This section is about understanding the data before doing model comparison. By the end of it, you should have a clear sense of how each modality is organized, which signals appear reliable, and how the provided reliability estimates relate to your own computations.

## 1.1 Inspect the datasets

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Inspect the content and axis meaning of TVSD, EEG2, and NSD.
- Identify where subject IDs, ROI labels, time axes, stimulus IDs, and noise ceilings are stored.
- State clearly what each axis means for each response array.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **TVSD structure table**
2. **EEG2 structure table**
3. **NSD structure table**

Each table must list the array name, array shape, and the meaning of each axis.

In [ ]:
# TODO: inspect TVSD
# TODO: inspect EEG2
# TODO: inspect NSD
# TODO: summarize axis meaning

**Inspect TVSD**

In [ ]:
# Print number of groups in .h5 file
with h5py.File(TVSD_path, 'r') as f:
    keys = list(f.keys())
    print(f"Number of groups: {len(keys)}")

# Print .h5 file structure
#explore_h5(TVSD_path) # --> long output, uncomment to inspect the structure

**Inspect EEG2**

In [ ]:
# Print number of groups in .h5 file
with h5py.File(EEG2_path, 'r') as f:
    keys = list(f.keys())
    print(f"Number of groups: {len(keys)}")

# Print .h5 file structure
#explore_h5(EEG2_path) # --> Long output, uncomment to inspect the structure


**Inspect NSD**

In [ ]:
# Define data paths
NSD_path = "/shared/NX-414/data/nsd_func1pt8mm_individualROIs.h5"

# Print number of groups in .h5 file
with h5py.File(NSD_path, 'r') as f:
    keys = list(f.keys())
    print(f"Number of groups: {len(keys)}")

# Print .h5 file structure
#explore_h5(NSD_path) #--> Long output, uncomment to inspect the structure


### TVSD

| Array | Shape | Axis 0 | Axis 1 |
|-------|-------|--------|--------|
| `train/neural_data/monkeyF/IT` | (22248, 241) | image presentations | IT neurons |
| `train/neural_data/monkeyF/V1` | (22248, 462) | image presentations | V1 neurons |
| `train/neural_data/monkeyF/V4` | (22248, 139) | image presentations | V4 neurons |
| `train/neural_data/monkeyN/IT` | (22248, 178) | image presentations | IT neurons |
| `train/neural_data/monkeyN/V1` | (22248, 437) | image presentations | V1 neurons |
| `train/neural_data/monkeyN/V4` | (22248, 247) | image presentations | V4 neurons |
| `test/neural_data/monkeyF/IT` | (100, 241) | image presentations | IT neurons |
| `test/neural_data/monkeyF/V1` | (100, 462) | image presentations | V1 neurons |
| `test/neural_data/monkeyF/V4` | (100, 139) | image presentations | V4 neurons |
| `test/neural_data/monkeyN/IT` | (100, 178) | image presentations | IT neurons |
| `test/neural_data/monkeyN/V1` | (100, 437) | image presentations | V1 neurons |
| `test/neural_data/monkeyN/V4` | (100, 247) | image presentations | V4 neurons |
| `train/stimulus_ids` | (22248,) | image presentations | — |
| `test/stimulus_ids` | (100,) | image presentations | — |
| `noise_ceilings/monkeyF/IT` | (241,) | IT neurons | — |
| `noise_ceilings/monkeyF/V1` | (462,) | V1 neurons | — |
| `noise_ceilings/monkeyF/V4` | (139,) | V4 neurons | — |
| `noise_ceilings/monkeyN/IT` | (178,) | IT neurons | — |
| `noise_ceilings/monkeyN/V1` | (437,) | V1 neurons | — |
| `noise_ceilings/monkeyN/V4` | (247,) | V4 neurons | — |

- **Subject IDs:** encoded in path (`monkeyF`, `monkeyN`)
- **ROI labels:** encoded in path (`IT`, `V1`, `V4`)
- **Noise ceilings:** one reliability score per neuron

---

### EEG2

| Array | Shape | Axis 0 | Axis 1 | Axis 2 |
|-------|-------|--------|--------|--------|
| `train/neural_data/sub-XX/region` | (16540, n_ch, 80) | image presentations | EEG channels | timepoints |
| `test/neural_data/sub-XX/region` | (200, n_ch, 80) | image presentations | EEG channels | timepoints |
| `train/stimulus_ids` | (16540,) | image presentations | — | — |
| `test/stimulus_ids` | (200,) | image presentations | — | — |
| `noise_ceilings/sub-XX/region` | (n_ch, 80) | EEG channels | timepoints | — |
| `noise_ceilings_train/sub-XX/region` | (n_ch, 80) | EEG channels | timepoints | — |

- **Subjects:** `sub-01` → `sub-10`
- **Regions:** `central`, `frontal`, `occipital`, `occipital_parietal`, `parietal`, `temporal`, `whole_brain`
- **n_ch** varies by region (3–63 channels)
- **Time axis:** 80 timepoints per trial
- **Noise ceilings:** per-channel per-timepoint reliability scores

---

### NSD

| Array | Shape | Axis 0 | Axis 1 | Axis 2 |
|-------|-------|--------|--------|--------|
| `train/neural_data/subjXX/ROI` | (n_img, n_vox) | image presentations | fMRI voxels | — |
| `test/neural_data/subjXX/ROI` | (n_img, n_vox) | image presentations | fMRI voxels | — |
| `train/stimulus_ids/subjXX` | (n_img,) | image presentations | — | — |
| `test/stimulus_ids/subjXX` | (n_img,) | image presentations | — | — |
| `noise_ceilings/subjXX/ROI` | (n_vox,) | fMRI voxels | — | — |
| `noise_ceiling_full/subjXX` | (x, y, z) | MRI x-dim | MRI y-dim | MRI z-dim |
| `roi_labels/subjXX/ROI` | (x, y, z) | MRI x-dim | MRI y-dim | MRI z-dim |
| `roi_labels_nc/subjXX/ROI` | (x, y, z) | MRI x-dim | MRI y-dim | MRI z-dim |

- **Subjects:** `subj01` → `subj08`
- **n_img** varies per subject: 9000 / 8481 / 8302 (train), 1000 / 930 / 907 (test)
- **n_vox** varies per subject and ROI
- **ROIs:** 28 regions — `V1d/v`, `V2d/v`, `V3d/v`, `hV4`, `EBA`, `FFA-1/2`, `PPA`, `OPA`, `RSC`, `nsdgeneral`, `whole_brain`, and more
- **roi_labels / roi_labels_nc:** 3D boolean masks in native MRI space
- **noise_ceiling_full:** full-brain 3D reliability map in native MRI space

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.1</strong><br>Explain the main differences between the three modalities in terms of what is being measured and how the data are organized.</div>
The three modalities measure different neural signals. TVSD contains spiking neural responses from two monkeys across three visual areas (IT, V1, V4), with no explicit time axis and data organized as image presentations × neurons, providing high spatial resolution at the single-neuron level. EEG2 contains EEG responses from 10 human subjects with a full time axis (80 time points) across 7 scalp regions, structured as stimuli × channels × time, and offering high temporal resolution. NSD contains fMRI BOLD responses from 8 human subjects across 28 cortical ROIs, with subject-specific voxel counts and additional 3D MRI-based representations (e.g., ROI masks and full-brain volumes), providing high spatial resolution at the voxel level. All datasets include stimulus IDs to align neural responses with image presentations and provide noise ceiling estimates as reliability measures. The key structural differences are: TVSD has no temporal axis (responses are averaged over a time window centered around the peak of activity), EEG2 adds a temporal dimension, and NSD has the richest spatial structure (voxels and ROIs) but no time axis.

---

## 1.2 Visualize EEG signals

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Plot example EEG responses for several stimuli and channels.
- Plot average responses over time for at least one subject and one ROI.
- Visualize the provided EEG noise ceilings over channels and time.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One plot of example EEG time courses** for several stimuli and channels.
2. **One heatmap over channels × time** for at least one subject and one ROI.
3. **One summary plot of the provided EEG noise ceilings**.
4. **One short written interpretation** in Answer box 1.2.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.2</strong><br>Which time windows and channel groups appear most informative? Are the responses dominated by noise, or do you observe clear evoked structure?</div>
Looking at the heatmap generated for subject 1 (central ROI) the most significant time window appears to be between 0 and 0.5 seconds, where neural activity has a more clear oscillatory behavior, that resembles the stereotypical spiking activity. The channels where this stimulus-evoked oscillations are more evident are channels 1 and 3, and more weakly the same activity pattern is observable in channels 8, 9, 11, 12. On the other hand, the remaining channels (0, 2, 4, 5, 6, 7, 10, 13) are noisier, since they do not highlight any particular oscillatory activity. These observations are supported by the noise ceilings heatmap, where higher oscillatory evoked activity is present in channels 1 and 3 in the first 0.5 seconds.

In [ ]:
# TODO: load one EEG subject / ROI
# TODO: plot example traces
# TODO: plot channel x time heatmap
# TODO: visualize provided EEG noise ceilings

### 1.2 EEG visualization — 4 points
- 1 pt: example EEG time-course plot is present and readable.
- 1 pt: channel × time heatmap is present and readable.
- 1 pt: provided EEG noise ceiling visualization is present and readable.
- 1 pt: written interpretation identifies informative time windows or channel groups.


**Loading EEG**

In [ ]:
# Load one EEG subject / ROI
with h5py.File(EEG2_path, 'r') as f:
    subs = list(f['train/neural_data'].keys())
    rois = list(f[f'train/neural_data/{subs[0]}'].keys())

sub = subs[0]   # first available subject
roi = rois[0]   # first available ROI

with h5py.File(EEG2_path, 'r') as f:
    eeg_data   = f[f'train/neural_data/{sub}/{roi}'][:]   # (n_stimuli, n_channels, n_timepoints)
    noise_ceil = f[f'noise_ceilings/{sub}/{roi}'][:]      # (n_channels, n_timepoints)

n_stimuli, n_channels, n_timepoints = eeg_data.shape
times = np.arange(n_timepoints) / 100.0

print(f'Subject: {sub}   |   ROI : {roi}')
print(f'EEG data: {eeg_data.shape}  (stimuli × channels × time)')
print(f'Noise ceiling: {noise_ceil.shape}  (channels × time)')
print(f'Time axis: {times[0]:.2f} – {times[-1]:.2f} s  ({n_timepoints} pts @ 100 Hz)')


**EEG time course for stimuli and channels**

In [ ]:
# Plot example EEG time courses for several stimuli and channels
n_stim = 5
n_ch = 5

rng = np.random.default_rng(42)
s_idx  = rng.choice(n_stimuli, n_stim, replace=False)
ch_idx = np.linspace(0, n_channels - 1, n_ch, dtype=int)

fig, axes = plt.subplots(n_ch, 1, figsize=(10, 2.2 * n_ch), sharex=True)
palette   = plt.cm.tab10(np.linspace(0, 0.9, n_stim))

for row, ch in enumerate(ch_idx):
    ax = axes[row]
    for k, si in enumerate(s_idx):
        ax.plot(times, eeg_data[si, ch, :], color=palette[k], lw=1.0, alpha=0.85,
                label=f'stim {si}' if row == 0 else '_')
    ax.axhline(0, color='0.7', lw=0.6)
    ax.set_ylabel(f'Ch {ch}', fontsize=8)

axes[0].legend(ncol=3, fontsize=7, loc='upper right')
axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Example EEG time courses — {sub} / {roi}', fontsize=11)
plt.tight_layout()
plt.show()


![](EEG.png)

**Heatmap over channels and time, example traces**

In [ ]:
# Channel × time heatmap of EEG response averaged over stimuli
mean_resp = eeg_data.mean(axis=0)   #(n_ch, n_t)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(mean_resp, aspect='auto', origin='lower',
               extent=[times[0], times[-1], 0, n_channels],
               cmap='RdBu_r')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Channel index')
ax.set_title(f'Mean EEG response — {sub} / {roi}  (n = {n_stimuli} stimuli)')
plt.colorbar(im, ax=ax, label='Amplitude (a.u.)')
plt.tight_layout()
plt.show()


![](MeanEEG.png)

**Summary plot of noise ceilings**

In [ ]:
# Visualize provided EEG noise ceilings
nc = noise_ceil / 100.0
channels = noise_ceil.shape[0]
ax_ch = np.linspace(0, channels-1, channels)


fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Left: channels × time heatmap
ax = axes[0]
im = ax.imshow(nc, aspect='auto', origin='lower',
               extent=[times[0], times[-1], 0, n_channels],
               cmap='viridis', vmin=0, vmax=nc.max())
ax.set_xlabel('Time (s)')
ax.set_ylabel('Channel index')
ax.set_title(f'Noise ceiling — {sub} / {roi}')
plt.colorbar(im, ax=ax, label='Noise ceiling [0–1]')

# Middle: mean ± std over channels in time
mu  = nc.mean(axis=0)
sig = nc.std(axis=0)
ax  = axes[1]
ax.plot(times, mu, lw=2, color='steelblue', label='mean ± std across channels')
ax.fill_between(times, mu - sig, mu + sig, alpha=0.25, color='steelblue')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Noise ceiling')
ax.set_title(f'Mean noise ceiling over channels — {sub} / {roi}')
ax.set_ylim(0)
ax.legend(fontsize=9)

# Right: mean ± std over time in channels
mu_tp  = nc.mean(axis=1)
sig_tp = nc.std(axis=1)
ax  = axes[2]
#ax.bar(ax_ch, mu_tp, lw=2, color='steelblue', label='mean ± std across channels')
#ax.fill_between(ax_ch, mu_tp - sig_tp, mu_tp + sig_tp, alpha=0.25, color='steelblue')
ax.bar(ax_ch, mu_tp, yerr=sig_tp, width=1, color='steelblue', 
       edgecolor='steelblue', alpha=0.7, label='mean ± std across channels',
       error_kw={'ecolor': 'black', 'lw': 1, 'alpha': 0.5})
ax.set_xlabel('Time (s)')
ax.set_ylabel('Noise ceiling')
ax.set_title(f'Mean noise ceiling over time — {sub} / {roi}')
ax.set_ylim(0)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


![](noise.png)

---

## 1.3 Estimate EEG noise ceilings using two methods

In practice, there are multiple ways to estimate noise ceilings, depending on the available data and the specific research question. When you have repeated measurements of the same stimulus, you can estimate reliability from the consistency of those repetitions. When repeated measurements are not available, reliability can instead be estimated across subjects, which often yields a more conservative ceiling.

In this part, you will implement two different estimators using the `things_eeg2-test_reps.h5` file, which contains the unaveraged test responses with repetitions.

You must implement **two different estimators** using `things_eeg2-test_reps.h5`. You can refer to the cited paper in each method's docstring for details.


### Required estimators

1. **Variance-based estimator**  
2. **Split-half reliability estimator**

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Implement both estimators.
- Compute both estimators for EEG2.
- Compare them to the provided EEG noise ceilings stored in `things_eeg2.h5`.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Working implementation of the variance-based estimator**
2. **Working implementation of the split-half estimator**
3. **One plot of mean noise ceiling over time**
4. **One plot of mean noise ceiling over channels**
5. **At least one channel × time heatmap for each estimator**
6. **At least one histogram comparing the value distributions**
7. **One direct visual comparison to the stored EEG noise ceilings**

### Starter functions

In [ ]:
import numpy as np


def compute_ceiling_variancebased(responses: np.ndarray, nan_policy: str = 'omit') -> np.ndarray:
    """
    Noise ceiling per unit using the method described in the NSD paper
    (Allen et al., 2021 / 2022 style variance-based estimator).

    Steps:
      1) z-score across stimuli (axis=1) for each (unit, rep) -> total var ≈ 1
      2) estimate noise variance across repetitions (axis=2), then average across stimuli
      3) signal variance = 1 - noise_var
      4) reliability (percent) for finite repeats:
             nc = 100 * (snr / (snr + 1 / n_reps))

    Parameters
    ----------
    responses : np.ndarray
        Shape (n_units, n_stimuli, n_reps) or
        (n_channels, n_timepoints, n_stimuli, n_reps).
    nan_policy : {'propagate', 'raise', 'omit'}, default='omit'
        Passed to the z-scoring logic when handling NaNs.

    Returns
    -------
    np.ndarray
        Per-unit noise ceilings in percent with shape (n_units,) or
        (n_channels, n_timepoints), depending on your implementation.
    """

    if responses.ndim == 4:
        n_channels, n_timepoints, n_stimuli, n_reps = responses.shape
        # process each (channel, timepoint) unit independently
        X = responses.reshape(n_channels * n_timepoints, n_stimuli, n_reps)

    # TVSD: (n_units, n_stimuli, n_reps)
    elif responses.ndim == 3:
        n_units, n_stimuli, n_reps = responses.shape
        X = responses.copy()

    else:
        raise ValueError(f"Expected 3D or 4D array, got shape {responses.shape}")

    n_units, n_stimuli, n_reps = X.shape

    # Step 1: z-score across stimuli for each (unit, rep)
    if nan_policy == 'omit':
        mean = np.nanmean(X, axis=1, keepdims=True)
        std  = np.nanstd( X, axis=1, keepdims=True)
    elif nan_policy == 'propagate':
        mean = X.mean(axis=1, keepdims=True)
        std  = X.std( axis=1, keepdims=True)
    elif nan_policy == 'raise':
        if np.any(np.isnan(X)):
            raise ValueError("NaNs found in responses and nan_policy='raise'")
        mean = X.mean(axis=1, keepdims=True)
        std  = X.std( axis=1, keepdims=True)
    else:
        raise ValueError(f"Unknown nan_policy: {nan_policy!r}")

    std = np.where(std == 0, np.nan, std)
    Z = (X - mean) / std                              # (n_units, n_stimuli, n_reps)

    # Step 2: noise variance across reps, averaged across stimuli
    if nan_policy == 'omit':
        noise_var = np.nanmean(np.nanvar(Z, axis=2, ddof=1), axis=1)
    else:
        noise_var = np.mean(np.var(Z, axis=2, ddof=1), axis=1)

    # Step 3: signal variance
    signal_var = np.maximum(1.0 - noise_var, 0)

    # Step 4: finite-repeat correction -> (n_units,)
    nc = 100.0 * (signal_var / (signal_var + 1.0 / n_reps))

    # Reshape output to match input structure
    if responses.ndim == 4:
        return nc.reshape(n_channels, n_timepoints)   # EEG: (n_channels, n_timepoints)
    else:
        return nc                                      # TVSD: (n_units,)


def compute_ceiling_splithalf(
    responses: np.ndarray,
    folds: int = 10,
    seed: int = 0,
    spearman_brown: bool = True,
    equalize_halves: bool = True,
    clip_folds: bool = False
) -> np.ndarray:
    """
    Split-half reliability per unit (voxel / channel / channel*timepoint).
    You can refer to van Bree et al. (2025) for mathematical details.
    
    Steps:
      1) For each fold, randomly split repetitions into two halves.
      2) Average responses within each half and compute Pearson correlation across stimuli.
      3) Optionally apply Spearman-Brown correction to each fold's correlation.
      4) Average across folds to get a final reliability estimate.

    Parameters
    ----------
    responses : np.ndarray
        Shape (n_units, n_stimuli, n_reps) or
        (n_channels, n_timepoints, n_stimuli, n_reps).
        The last axis corresponds to repetitions / trials.
    folds : int, default=10
        Number of random split-halves to sample.
    seed : int, default=0
        Base RNG seed; each fold may use seed + fold_idx.
    spearman_brown : bool, default=True
        Apply Spearman-Brown correction:
            r_sb = 2r / (1 + r)
    equalize_halves : bool, default=True
        If True, use equal-sized halves and drop one trial if n_reps is odd.
        If False, the second half may be larger by one trial.
    clip_folds : bool, default=False
        If True, clip reliability values after correction.

    Returns
    -------
    np.ndarray
        Array of shape (n_units) or (n_channels, n_timepoints).
    """
    # EEG: (n_channels, n_timepoints, n_stimuli, n_reps)
    if responses.ndim == 4:
        n_channels, n_timepoints, n_stimuli, n_reps = responses.shape
        X = responses.reshape(n_channels * n_timepoints, n_stimuli, n_reps)

    # TVSD: (n_units, n_stimuli, n_reps)
    elif responses.ndim == 3:
        n_units, n_stimuli, n_reps = responses.shape
        X = responses.copy()

    else:
        raise ValueError(f"Expected 3D or 4D array, got shape {responses.shape}")

    n_units, n_stimuli, n_reps = X.shape
    half = n_reps // 2  # size of each half (drops one rep if n_reps is odd + equalize_halves)

    fold_corrs = np.zeros((folds, n_units))  # (folds, n_units)

    for fold in range(folds):
        rng = np.random.default_rng(seed + fold)
        perm = rng.permutation(n_reps)

        if equalize_halves:
            idx_a = perm[:half]
            idx_b = perm[half:2*half]   # drop last rep if n_reps is odd
        else:
            idx_a = perm[:half]
            idx_b = perm[half:]         # second half may be larger by one

        # Average across reps within each half -> (n_units, n_stimuli)
        half_a = X[:, :, idx_a].mean(axis=2)
        half_b = X[:, :, idx_b].mean(axis=2)

        # Step 2: Pearson correlation across stimuli for each unit
        # z-score each half across stimuli, then dot product
        def zscore(arr):
            mu  = arr.mean(axis=1, keepdims=True)
            std = arr.std( axis=1, keepdims=True)
            std = np.where(std == 0, np.nan, std)
            return (arr - mu) / std

        za = zscore(half_a)  # (n_units, n_stimuli)
        zb = zscore(half_b)  # (n_units, n_stimuli)

        r = np.nanmean(za * zb, axis=1)  # (n_units,) — Pearson r per unit

        # Step 3: Spearman-Brown correction
        if spearman_brown:
            r = 2 * r / (1 + r)

        if clip_folds:
            r = np.clip(r, 0, 1)

        fold_corrs[fold] = r

    # Step 4: average across folds -> (n_units,)
    nc = np.nanmean(fold_corrs, axis=0) * 100

    if responses.ndim == 4:
        return nc.reshape(n_channels, n_timepoints)
    else:
        return nc

In [ ]:
# TODO: load repeated EEG test responses
# TODO: compute variance-based estimator
# TODO: compute split-half estimator
# TODO: visualize and compare both estimators

**Load repeated EEG test responses**

In [ ]:
reps_path = "/shared/NX-414/data/things_eeg2-test_reps.h5"
#explore_h5(reps_path)

# Load EEG subjects and ROIs
with h5py.File(EEG2_path, 'r') as f:
    subs = list(f['test/neural_data'].keys())
    rois = list(f[f'test/neural_data/{subs[0]}'].keys())

nc_var = {roi: {} for roi in rois}  # {roi: {sub: (n_channels, n_timepoints)}}
nc_sh  = {roi: {} for roi in rois}

with h5py.File(reps_path, 'r') as f:
    for sub in subs:
        for roi in rois:
            data = f[f'test/neural_data/{sub}/{roi}'][:]  # (n_stimuli, n_channels, n_timepoints, n_reps)
            data = data.transpose(1, 2, 0, 3)             # -> (n_channels, n_timepoints, n_stimuli, n_reps)
            nc_var[roi][sub] = compute_ceiling_variancebased(data)
            nc_sh[roi][sub]  = compute_ceiling_splithalf(data, clip_folds=True)
        print(f"✓ {sub} done")

**Visualize and compare estimators (only whole-brain, sub-01)**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

sub = 'sub-01'
roi = 'whole_brain'

nc_var_s = nc_var[roi][sub]   # (n_channels, n_timepoints)
nc_sh_s  = nc_sh[roi][sub]

with h5py.File(EEG2_path, 'r') as f:
    nc_st_s = f[f'noise_ceilings/{sub}/{roi}'][:]

fig = plt.figure(figsize=(18, 20))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Mean noise ceiling over time (avg over channels)
ax = fig.add_subplot(gs[0, :])
for arr, label, color in [
    (nc_var_s, 'Variance-based', 'steelblue'),
    (nc_sh_s,  'Split-half',     'tomato'),
    (nc_st_s,  'Stored',         'seagreen'),
]:
    ax.plot(arr.mean(axis=0), label=label, color=color, linewidth=2)
ax.set_xlabel('Timepoint')
ax.set_ylabel('Noise ceiling (%)')
ax.set_title(f'Mean noise ceiling over time — {sub} {roi}')
ax.legend()

# 2. Mean noise ceiling over channels (avg over time)
ax = fig.add_subplot(gs[1, :])
x = np.arange(nc_var_s.shape[0])
w = 0.25
for i, (arr, label, color) in enumerate([
    (nc_var_s, 'Variance-based', 'steelblue'),
    (nc_sh_s,  'Split-half',     'tomato'),
    (nc_st_s,  'Stored',         'seagreen'),
]):
    ax.bar(x + (i-1)*w, arr.mean(axis=1), width=w, label=label, color=color, alpha=0.8)
ax.set_xlabel('Channel')
ax.set_ylabel('Noise ceiling (%)')
ax.set_title(f'Mean noise ceiling over channels — {sub} {roi}')
ax.legend()

# 3. Heatmaps: channel x time
vmin = min(nc_var_s.min(), nc_sh_s.min(), nc_st_s.min())
vmax = max(nc_var_s.max(), nc_sh_s.max(), nc_st_s.max())

for col, (arr, title) in enumerate([
    (nc_var_s, 'Variance-based'),
    (nc_sh_s,  'Split-half'),
    (nc_st_s,  'Stored'),
]):
    ax = fig.add_subplot(gs[2, col])
    im = ax.imshow(arr, aspect='auto', origin='lower', vmin=vmin, vmax=vmax, cmap='viridis')
    ax.set_xlabel('Timepoint')
    ax.set_ylabel('Channel')
    ax.set_title(f'Heatmap: {title}')
    plt.colorbar(im, ax=ax, label='NC (%)')

![](nc_comp.png)

**Visualize and compare estimators (only whole-brain, mean across subjects)**

In [ ]:
roi = 'whole_brain'

with h5py.File(EEG2_path, 'r') as f:
    subs = list(f['test/neural_data'].keys())
    
# Stack across subjects -> (n_subjects, n_channels, n_timepoints)
nc_var_arr = np.stack([nc_var[roi][sub] for sub in subs])
nc_sh_arr  = np.stack([nc_sh[roi][sub]  for sub in subs])
    
# Load stored noise ceilings -> (n_subjects, n_channels, n_timepoints)
with h5py.File(EEG2_path, 'r') as f:
    nc_stored_arr = np.stack([
        f[f'noise_ceilings/{sub}/{roi}'][:] for sub in subs
    ])

# Mean and std across subjects
nc_var_mean, nc_var_std = nc_var_arr.mean(0), nc_var_arr.std(0)   # (n_channels, n_timepoints)
nc_sh_mean,  nc_sh_std  = nc_sh_arr.mean(0),  nc_sh_arr.std(0)
nc_st_mean,  nc_st_std  = nc_stored_arr.mean(0), nc_stored_arr.std(0)

fig = plt.figure(figsize=(18, 20))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Mean noise ceiling over time (avg over channels)
ax = fig.add_subplot(gs[0, :])
for arr, std, label, color in [
    (nc_var_mean, nc_var_std, 'Variance-based', 'steelblue'),
    (nc_sh_mean,  nc_sh_std,  'Split-half',     'tomato'),
    (nc_st_mean,  nc_st_std,  'Stored',         'seagreen'),
]:
    mu = arr.mean(axis=0)        # (n_timepoints,)
    se = std.mean(axis=0)
    ax.plot(mu, label=label, color=color, linewidth=2)
    ax.fill_between(range(len(mu)), mu - se, mu + se, alpha=0.2, color=color)
ax.set_xlabel('Timepoint')
ax.set_ylabel('Noise ceiling (%)')
ax.set_title('Mean noise ceiling over time (avg across channels ± std)')
ax.legend()

# 2. Mean noise ceiling over channels (avg over time)
ax = fig.add_subplot(gs[1, :])
x = np.arange(nc_var_mean.shape[0])
w = 0.25
for i, (arr, std, label, color) in enumerate([
    (nc_var_mean, nc_var_std, 'Variance-based', 'steelblue'),
    (nc_sh_mean,  nc_sh_std,  'Split-half',     'tomato'),
    (nc_st_mean,  nc_st_std,  'Stored',         'seagreen'),
]):
    mu = arr.mean(axis=1)   # (n_channels,)
    se = std.mean(axis=1)
    ax.bar(x + (i-1)*w, mu, width=w, label=label, color=color, alpha=0.8)
    ax.errorbar(x + (i-1)*w, mu, yerr=se, fmt='none', color='black', capsize=2, linewidth=0.8)
ax.set_xlabel('Channel')
ax.set_ylabel('Noise ceiling (%)')
ax.set_title('Mean noise ceiling over channels (avg across time ± std)')
ax.legend()

# 3. Heatmaps: channel x time
vmin = min(nc_var_mean.min(), nc_sh_mean.min(), nc_st_mean.min())
vmax = max(nc_var_mean.max(), nc_sh_mean.max(), nc_st_mean.max())

for col, (arr, title) in enumerate([
    (nc_var_mean, 'Variance-based'),
    (nc_sh_mean,  'Split-half'),
    (nc_st_mean,  'Stored'),
]):
    ax = fig.add_subplot(gs[2, col])
    im = ax.imshow(arr, aspect='auto', origin='lower', vmin=vmin, vmax=vmax, cmap='viridis')
    ax.set_xlabel('Timepoint')
    ax.set_ylabel('Channel')
    ax.set_title(f'Heatmap: {title}')
    plt.colorbar(im, ax=ax, label='NC (%)')

# 4. Histogram
ax = fig.add_subplot(gs[3, 0])
for arr, label, color in [
    (nc_var_arr.ravel(),    'Variance-based', 'steelblue'),
    (nc_sh_arr.ravel(),     'Split-half',     'tomato'),
    (nc_stored_arr.ravel(), 'Stored',         'seagreen'),
]:
    ax.hist(arr, bins=60, alpha=0.5, label=label, color=color, density=True)
ax.set_xlabel('Noise ceiling (%)')
ax.set_ylabel('Density')
ax.set_title('Distribution of noise ceiling values (all subjects, channels, timepoints)')
ax.legend()

# 5a. Scatter: variance-based vs stored
ax = fig.add_subplot(gs[3, 1])
ax.scatter(nc_st_mean.ravel(), nc_var_mean.ravel(), alpha=0.3, s=5, color='steelblue')
lims = [min(nc_st_mean.min(), nc_var_mean.min()), max(nc_st_mean.max(), nc_var_mean.max())]
ax.plot(lims, lims, 'k--', linewidth=1)
ax.set_xlabel('Stored NC')
ax.set_ylabel('Estimated NC')
ax.set_title('Variance-based vs stored')

# 5b. Scatter: split-half vs stored
ax = fig.add_subplot(gs[3, 2])
ax.scatter(nc_st_mean.ravel(), nc_sh_mean.ravel(), alpha=0.3, s=5, color='tomato')
lims = [min(nc_st_mean.min(), nc_sh_mean.min()), max(nc_st_mean.max(), nc_sh_mean.max())]
ax.plot(lims, lims, 'k--', linewidth=1)
ax.set_xlabel('Stored NC')
ax.set_ylabel('Estimated NC')
ax.set_title('Split-half vs stored')
plt.suptitle(f'Noise ceiling comparison — {roi}', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('noise_ceiling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

![](noise_ceiling_comparison.png)

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.3</strong><br>Compare the two estimators. Do they produce similar patterns across channels and time? Where do they differ most?</div>
The two estimators show a similar pattern to the stored noise ceilings overtime, with the split-half estimate that drifts after the 50th time point. Over channels the difference in more evident and seems that the variance-based estimate is more reliable than the split-half. Also from the distribution plot and the two scatterplot, the overlap between the variance-based estimator and the stored ceilings is evident (while the split-half estimator is more spread around y=x)

---

## 1.4 Compare the noise ceiling estimators statistically on EEG2

In `things_eeg2.h5`, we provided noise ceilings computed using one of the two methods you implemented. Can you determine which one it is by comparing the stored ceilings to your computed ones?

Perform a hypothesis test to compare the stored ceilings to each of your computed estimators. For example, you could compute the mean squared error between the stored ceilings and each estimator per subject/time/channel, and then use a paired t-test to see if one estimator is significantly closer to the stored values than the other.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- State clearly what each estimator assumes.
- Define a quantitative comparison to the stored EEG noise ceilings.
- Run at least one simple statistical test or formal comparison.

Examples:

- mean absolute deviation from the stored values,
- paired comparison across channel × time units,
- correlation with the stored values.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One quantitative comparison table** comparing both estimators to the stored EEG noise ceilings.
2. **One statistical test or one formal quantitative comparison** such as a paired test, correlation analysis, or mean absolute deviation analysis.
3. **One concise written conclusion** stating which estimator better matches the stored values and why.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.4</strong><br>Which estimator is more likely to have been used to generate the stored EEG noise ceilings? Justify your answer with both visual and quantitative evidence.</div>
The variance-based estimator is most likely the one used to generate the stored EEG noise ceilings. This is supported by both visual and quantitative evidence. Visually, the time/channel plots show that the variance-based estimator tracks the stored ceilings almost perfectly, while the split-half estimator, although correlated, shows a systematic upward bias and higher spread. Quantitatively, the comparison table confirms this: the variance-based estimator achieves a near-perfect mean correlation of 0.9999 and a mean MSE of 0.087, while the split-half estimator, despite a still high correlation of 0.983, has a much larger mean MSE of 13.32; To formally compare the two estimators, we ran a paired t-test on the squared errors (per channel × timepoint unit) of each estimator against the stored ceilings. The paired design is appropriate here because each unit (channel × timepoint) provides one squared error for each estimator, making the samples naturally paired. The test yielded t = −33.95 and p = 2.36e−251, confirming that the variance-based estimator has significantly lower squared error than the split-half estimator. The extremely small p-value leaves no doubt that the difference is not due to chance.

In [ ]:
# TODO: define estimator comparison metric
# TODO: run statistical comparison
# TODO: summarize which estimator best matches stored values

**Load stored values**

In [ ]:
stored_nc = {}
with h5py.File(EEG2_path, "r") as f:
    subs = list(f["noise_ceilings"].keys())
    for sub in subs:
        stored_nc[sub] = {}
        rois = list(f["noise_ceilings"][sub].keys())
        for roi in rois:
            stored_nc[sub][roi] = f["noise_ceilings"][sub][roi][()]

In [ ]:
stored_nc[sub][roi] = stored_nc[sub][roi]/100
nc_sh[roi][sub] = nc_sh[roi][sub]/100
nc_var[roi][sub] = nc_var[roi][sub]/100

**Quantitative metrics across all subjects and ROIs**

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, ttest_rel

rows = []

all_sqerr_var = []
all_sqerr_sh = []

for sub in stored_nc:
    for roi in stored_nc[sub]:
        stored = stored_nc[sub][roi]
        var_est = nc_var[roi][sub]
        sh_est = nc_sh[roi][sub]

        # check shape
        assert stored.shape == var_est.shape == sh_est.shape, f"Shape mismatch in {sub} {roi}"

        # flatten
        s = stored.reshape(-1)
        v = var_est.reshape(-1)
        sh = sh_est.reshape(-1)

        # metrics
        mse_var = np.mean((s - v) ** 2)
        mae_var = np.mean(np.abs(s - v))
        corr_var = pearsonr(s, v)[0] if len(s) > 1 else np.nan

        mse_split = np.mean((s - sh) ** 2)
        mae_split = np.mean(np.abs(s - sh))
        corr_split = pearsonr(s, sh)[0] if len(s) > 1 else np.nan

        rows.append({
            "subject": sub,
            "roi": roi,
            "mse_var": mse_var,
            "mae_var": mae_var,
            "corr_var": corr_var,
            "mse_split": mse_split,
            "mae_split": mae_split,
            "corr_split": corr_split
        })

        # save squared errors for paired test
        all_sqerr_var.extend((s - v) ** 2)
        all_sqerr_sh.extend((s - sh) ** 2)

df_compare = pd.DataFrame(rows)
df_compare.head()

| subject | roi | mse_var | mae_var | corr_var | mse_split | mae_split | corr_split |
|---------|-----|---------|---------|----------|-----------|-----------|------------|
| sub-01 | central | 0.004411 | 0.033415 | 0.999999 | 17.602268 | 2.396104 | 0.941141 |
| sub-01 | frontal | 0.001004 | 0.016845 | 0.999999 | 10.026428 | 2.439059 | 0.932257 |
| sub-01 | occipital | 0.098507 | 0.201947 | 0.999997 | 6.200087 | 1.806519 | 0.995778 |
| sub-01 | occipital_parietal | 0.058615 | 0.150501 | 0.999998 | 75.788389 | 3.223022 | 0.935724 |
| sub-01 | parietal | 0.050067 | 0.139477 | 0.999998 | 90.700168 | 3.526559 | 0.917501 |

**Quantitative comparison table**

In [ ]:
summary_table = pd.DataFrame({
    "Estimator": ["Variance-based", "Split-half"],
    "Mean MSE": [df_compare["mse_var"].mean(), df_compare["mse_split"].mean()],
    "Mean MAE": [df_compare["mae_var"].mean(), df_compare["mae_split"].mean()],
    "Mean Correlation": [df_compare["corr_var"].mean(), df_compare["corr_split"].mean()]
})

summary_table

| Estimator | Mean MSE | Mean MAE | Mean Correlation |
|-----------|----------|----------|-----------------|
| Variance-based | 0.087543 | 0.168459 | 0.999997 |
| Split-half | 13.323223 | 2.063192 | 0.983310 |

**Statistical test (paired t-test)**

In [ ]:
t_stat, p_val = ttest_rel(all_sqerr_var, all_sqerr_sh)

print("Paired t-test on squared errors")
print("t =", t_stat)
print("p =", p_val)

| Test | t-statistic | p-value |
|------|-------------|---------|
| Paired t-test on squared errors | -33.9509 | 2.358e-251 |

---

## 1.5 Convert NSD ncsnr to noise ceiling and visualize it on cortex

Some datasets, such as NSD, provide reliability estimates with the data release. In this section, you will visualize the provided NSD reliability estimates on the cortical surface and convert them into noise ceilings for later use in predictive analyses.

The provided NSD reliability estimates are stored as **ncsnr** values on the fsaverage surface. To use them as noise ceilings for voxel-wise analyses, you need to convert ncsnr to noise ceiling using the formula provided in the NSD methods paper.

Parcellations and atlases provide group-level anatomical labels for brain regions. They are often defined on a standard surface or volume space (e.g., fsaverage, MNI) and can be used to summarize or interpret neural data. For this exercise, use the Destrieux atlas to anatomically label the regions with the highest and lowest noise ceilings. It is available in fsaverage space and can be accessed through `nilearn`. Compute the average noise ceiling within each atlas region and identify which regions have the highest and lowest reliability.

If the available atlas is in a different surface resolution (e.g. `fsaverage5`), you can interpolate either the atlas or the noise ceiling map to the same space before visualization. Prefer downsampling rather than upsampling to avoid introducing artificial precision.

You can use `nibabel` to load the `.mgh` files and `nilearn` to visualize the resulting noise ceiling on the fsaverage surface.


<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Load the provided `.mgh` files for subject 01.
- Convert **ncsnr** to a **noise ceiling estimate** using the formula described in the NSD paper.
- Visualize the resulting noise ceiling on the fsaverage surface.
- Overlay a cortical parcellation.
- Compute parcel-wise average values.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One histogram of ncsnr values**
2. **One cortical surface plot** of ncsnr or the derived noise ceiling
3. **One cortical surface plot with parcel overlay**
4. **One parcel-wise summary figure or one parcel-wise summary table**

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 1.5</strong><br>Which cortical regions appear most reliable, and which appear least reliable? Explain how the parcellation helps interpret the surface maps.</div>
The most reliable cortical regions are concentrated in *early and higher visual cortex*: occipital areas (V1/V2/V3) consistently show the highest noise ceilings. These areas are strongly and consistently driven by the visual images in NSD, so repeated presentations of the same image elicit highly similar fMRI responses. Parts of the parietal cortex—particularly around the intraparietal sulcus—also show moderate reliability, reflecting their role in visuospatial processing.

The least reliable regions are frontal and non-visual cortices: anterior frontal areas, the orbitofrontal cortex, and the frontal pole have noise ceilings near zero because they are not systematically modulated by the visual stimuli. The medial wall (cingulate, motor cortex) and temporal pole similarly show very low values.

The Destrieux parcellation helps interpret the surface maps in two complementary ways. First, it provides anatomical labels that identify which specific gyri and sulci are most reliable, moving beyond an unlabeled gradient and allowing neuroscientifically meaningful comparisons. Second, computing parcel-wise averages reduces vertex-level noise, producing a cleaner regional summary that is easier to compare across subjects or datasets and to relate to known functional anatomy.

In [ ]:
# TODO: load lh/rh ncsnr maps
# TODO: convert ncsnr to noise ceiling
# TODO: plot histogram
# TODO: visualize on fsaverage
# TODO: compute parcel-wise summary

**Loading lh/rh ncsnr maps**

In [ ]:
# Load lh/rh ncsnr maps for subject 01
lh_path = "/shared/NX-414/data/nsd-subj01-ncsnr-lh.mgh"
rh_path = "/shared/NX-414/data/nsd-subj01-ncsnr-rh.mgh"

ncsnr_lh = nib.load(lh_path).get_fdata().squeeze()   # (n_vertices,)
ncsnr_rh = nib.load(rh_path).get_fdata().squeeze()

print(f"LH ncsnr: shape={ncsnr_lh.shape}, range=[{ncsnr_lh.min():.3f}, {ncsnr_lh.max():.3f}]")
print(f"RH ncsnr: shape={ncsnr_rh.shape}, range=[{ncsnr_rh.min():.3f}, {ncsnr_rh.max():.3f}]")
print(f"Non-zero vertices — LH: {(ncsnr_lh > 0).sum()}, RH: {(ncsnr_rh > 0).sum()}")

**Converting ncsnr to noise ceiling**

In [ ]:
# NSD paper formula:
#   NC = ncsnr^2 / (ncsnr^2 + 1/n_reps)
# n_reps = 3: average number of image repetitions per stimulus in NSD
n_reps = 3

nc_lh = ncsnr_lh**2 / (ncsnr_lh**2 + 1.0 / n_reps)
nc_rh = ncsnr_rh**2 / (ncsnr_rh**2 + 1.0 / n_reps)

# Masks for in-brain vertices (exclude medial wall where ncsnr == 0)
valid_lh = ncsnr_lh > 0
valid_rh = ncsnr_rh > 0

print(f"LH noise ceiling: mean={nc_lh[valid_lh].mean():.4f}, max={nc_lh.max():.4f}")
print(f"RH noise ceiling: mean={nc_rh[valid_rh].mean():.4f}, max={nc_rh.max():.4f}")


**Plot histogram**

In [ ]:
# Histogram of ncsnr values and the derived noise ceilings (in-brain vertices only)
all_ncsnr = np.concatenate([ncsnr_lh[valid_lh], ncsnr_rh[valid_rh]])
all_nc    = np.concatenate([nc_lh[valid_lh],    nc_rh[valid_rh]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, xlabel, color in [
    (axes[0], all_ncsnr, 'ncsnr',         'steelblue'),
    (axes[1], all_nc,    'noise ceiling', 'darkorange'),
]:
    ax.hist(data, bins=80, color=color, edgecolor='none', alpha=0.85)
    med = np.median(data)
    ax.axvline(med, color='k', lw=1.5, ls='--', label=f'median = {med:.3f}')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Vertex count')
    ax.set_title(f'Distribution of {xlabel} — subj01 (both hemispheres)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


**Visualize on fsaverage**

In [ ]:
from nilearn import datasets as ni_datasets
import nilearn.plotting as nlplt

fsaverage = ni_datasets.fetch_surf_fsaverage('fsaverage')

fig, axes = plt.subplots(2, 2, figsize=(16, 10), subplot_kw={'projection': '3d'})

panels = [
    ('left',  nc_lh, fsaverage['infl_left'],  fsaverage['sulc_left'],  'lateral'),
    ('left',  nc_lh, fsaverage['infl_left'],  fsaverage['sulc_left'],  'medial'),
    ('right', nc_rh, fsaverage['infl_right'], fsaverage['sulc_right'], 'lateral'),
    ('right', nc_rh, fsaverage['infl_right'], fsaverage['sulc_right'], 'medial'),
]

for ax, (hemi, nc_map, surf_mesh, bg_map, view) in zip(axes.ravel(), panels):
    nlplt.plot_surf_stat_map(
        surf_mesh, nc_map,
        bg_map=bg_map, hemi=hemi, view=view,
        colorbar=True, cmap='hot', vmin=0, vmax=1,
        title=f'{hemi} ({view})',
        axes=ax,
    )

fig.suptitle('NSD noise ceiling — subj01', fontsize=14, fontweight='bold')
plt.savefig('nsd_noise_ceiling_subj01.png', dpi=150, bbox_inches='tight')
plt.show()

![](nsd_noise_ceiling_subj01.png)

**Compute parcel-wise summary**

In [ ]:
from nilearn import surface as ni_surface
from scipy.spatial import cKDTree

fsaverage5 = ni_datasets.fetch_surf_fsaverage('fsaverage5')

# Downsample noise ceiling from fsaverage (163842 verts) to fsaverage5 (10242 verts)
# via nearest-neighbour lookup — preferred over upsampling

def nn_downsample(data, pial_src, pial_tgt):
    """
    Downsample surface data from a high-resolution mesh to a lower-resolution mesh
    via nearest-neighbour lookup.

    For each vertex in the target mesh, finds the closest vertex in the source mesh
    and assigns its value. Preferred over interpolation to avoid introducing
    artificial values between measured points.

    Parameters
    ----------
    data : np.ndarray
        Shape (n_vertices_src,) — data defined on the source mesh.
    pial_src : str
        Path to the source pial surface mesh (e.g. fsaverage, 163842 vertices).
    pial_tgt : str
        Path to the target pial surface mesh (e.g. fsaverage5, 10242 vertices).

    Returns
    -------
    np.ndarray
        Shape (n_vertices_tgt,) — data resampled onto the target mesh.
    """
    coords_src, _ = ni_surface.load_surf_mesh(pial_src)
    coords_tgt, _ = ni_surface.load_surf_mesh(pial_tgt)
    _, idx = cKDTree(coords_src).query(coords_tgt)
    return data[idx]

nc_lh_fsa5 = nn_downsample(nc_lh, fsaverage['pial_left'],  fsaverage5['pial_left'])
nc_rh_fsa5 = nn_downsample(nc_rh, fsaverage['pial_right'], fsaverage5['pial_right'])

# Load Destrieux atlas on fsaverage5
# labels_lh/rh: (10242,) integer parcel labels, one per vertex
destrieux    = ni_datasets.fetch_atlas_surf_destrieux()
labels_lh    = destrieux['map_left']
labels_rh    = destrieux['map_right']

# Convert integer labels to human-readable anatomical names
# .decode() handles bytes vs str inconsistencies across nilearn versions
parcel_names = [l.decode() if isinstance(l, bytes) else str(l)
                for l in destrieux['labels']]

# Compute per-parcel mean NC pooling both hemispheres
parcel_means = {}
for lbl in np.unique(np.concatenate([labels_lh, labels_rh])):

    # Label 0 = medial wall / unlabeled — not real cortex, skip
    if lbl == 0:
        continue

    # Collect ncsnr values for this parcel from both hemispheres
    vals = np.concatenate([
        nc_lh_fsa5[labels_lh == lbl],   # vertices in this parcel, left hemi
        nc_rh_fsa5[labels_rh == lbl],   # vertices in this parcel, right hemi
    ])

    # Remove out-of-mask vertices (ncsnr == 0 means outside fMRI volume)
    vals = vals[vals > 0]

    # Store mean only if at least one valid vertex exists in this parcel
    if len(vals) > 0:
        parcel_means[parcel_names[lbl]] = vals.mean()

# Map parcel means back to surface vertices for visualization
nc_parcel_lh = np.zeros(len(labels_lh))
nc_parcel_rh = np.zeros(len(labels_rh))
for lbl in np.unique(labels_lh):
    if lbl > 0 and parcel_names[lbl] in parcel_means:
        nc_parcel_lh[labels_lh == lbl] = parcel_means[parcel_names[lbl]]
for lbl in np.unique(labels_rh):
    if lbl > 0 and parcel_names[lbl] in parcel_means:
        nc_parcel_rh[labels_rh == lbl] = parcel_means[parcel_names[lbl]]

# Surface plot with Destrieux parcel overlay (parcels colored by mean NC)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), subplot_kw={'projection': '3d'})

for ax, (hemi, nc_map, surf_mesh, bg_map) in zip(axes, [
    ('left',  nc_parcel_lh, fsaverage5['infl_left'],  fsaverage5['sulc_left']),
    ('right', nc_parcel_rh, fsaverage5['infl_right'], fsaverage5['sulc_right']),
]):
    nlplt.plot_surf_stat_map(
        surf_mesh, nc_map,
        bg_map=bg_map, hemi=hemi, view='lateral',
        colorbar=True, cmap='hot', vmin=0, vmax=1,
        title=f'Parcel-wise NC (Destrieux) — subj01 {hemi}',
        axes=ax,
    )

fig.suptitle('Parcel-wise noise ceiling (Destrieux atlas) — subj01', fontsize=13, fontweight='bold')
plt.savefig('atlas_pw_nc.png', dpi=150, bbox_inches='tight')
plt.show()

# Bar chart: top and bottom 10 Destrieux regions by mean NC
N = 10
sorted_parcels = sorted(parcel_means.items(), key=lambda x: x[1], reverse=True)
top = sorted_parcels[:N]
bot = sorted_parcels[-N:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, regions, title, color in [
    (axes[0], top, f'Top {N} most reliable regions',    'forestgreen'),
    (axes[1], bot, f'Bottom {N} least reliable regions', 'firebrick'),
]:
    names, vals = zip(*regions)
    ax.barh(range(N), vals, color=color, alpha=0.8)
    ax.set_yticks(range(N))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel('Mean noise ceiling')
    ax.set_title(title)
    ax.set_xlim(0, 1)

plt.suptitle('Destrieux parcel-wise noise ceiling — subj01', fontsize=11)
plt.tight_layout()
plt.savefig('pw_nc.png', dpi=150, bbox_inches='tight')
plt.show()


![](atlas_pw_nc.png)

![](pw_nc.png)

---

# 2. Brain–Model Alignment

In this section, you will compare neural responses and model features using **both representational metrics and predictive linear models**. You must complete both parts of this section. The goal is not only to report scores, but also to compare what different metrics reveal about model–brain alignment.

## 2.1 Representational alignment: RSA

RSA stands for representational similarity analysis. It is one of the most widely used analyses in fMRI and model–brain alignment research. It compares the geometry of two representational spaces through their representational dissimilarity matrices (RDMs). Given two response matrices, `X` and `Y`, with rows corresponding to the same stimuli, we first compute an RDM for each matrix using correlation distance:

$$
D^X_{ij} = 1 - \mathrm{corr}(X[i,:], X[j,:]),
\qquad
D^Y_{ij} = 1 - \mathrm{corr}(Y[i,:], Y[j,:]),
$$

for stimulus pairs $i \neq j$.

We then vectorize the upper triangle of each RDM and compute RSA as the Spearman correlation between these two vectors:

$$
\mathrm{RSA}(X, Y)
=
\rho_{\mathrm{Spearman}}
\left(
\mathrm{vec}(D^X),\,
\mathrm{vec}(D^Y)
\right).
$$

In this project, `X` will usually denote model features from one candidate layer, and `Y` will denote neural responses from one dataset, ROI, subject, or time slice, depending on the analysis.

- Implement RSA between two representation matrices.
- Support at least one dissimilarity measure and one similarity measure.
- Use your implementation to compare model layers to neural responses.

### Starter code

In [ ]:
from typing import Literal
import numpy as np

class RepresentationalSimilarityAnalysis:
    """
    Representational Similarity Analysis (RSA).

    Given two representation matrices X and Y with the same number of conditions
    (rows), RSA:

    1. Computes a Representational Dissimilarity Matrix (RDM) for each:
       RDM_X[i, j] = dissimilarity(x_i, x_j)
       RDM_Y[i, j] = dissimilarity(y_i, y_j)

    2. Flattens the upper triangles of both RDMs and computes a correlation
       between them (Pearson or Spearman).
    """

    def __init__(
        self,
        dissimilarity: Literal["correlation", "euclidean", "cosine"] = "correlation",
        similarity_metric: Literal["pearson", "spearman"] = "spearman",
    ):
        ### TODO
        self.dissimilarity = dissimilarity
        self.similarity_metric = similarity_metric
        

    def __call__(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Compute RSA similarity between X and Y.

        Parameters
        ----------
        X, Y : np.ndarray
            Arrays of shape (n_conditions, ...) that may need to be flattened
            along feature dimensions.

        Returns
        -------
        rsa_similarity : float
            Correlation between the vectorized upper triangles of the two RDMs.
        """
        return self.forward(X, Y)

    def forward(self, X: np.ndarray, Y: np.ndarray) -> float:
        ### TODO
        #Flattening extra features to handle 2D matrices
        X_flat = X.reshape(X.shape[0], -1).astype(np.float64)
        Y_flat = Y.reshape(Y.shape[0], -1).astype(np.float64)

        rdm_x = self.compute_rdm(X_flat)
        rdm_y = self.compute_rdm(Y_flat)
        return self.compare_rdms(rdm_x, rdm_y)

    def compute_rdm(self, X: np.ndarray) -> np.ndarray:
        """
        Compute the Representational Dissimilarity Matrix (RDM)
        for a given representation matrix X.

        Parameters
        ----------
        X : np.ndarray
            Array of shape (n_conditions, n_features).

        Returns
        -------
        rdm : np.ndarray
            Array of shape (n_conditions, n_conditions) with pairwise dissimilarities.
        """
        ### TODO
        X_flat = X.reshape(X.shape[0], -1).astype(np.float64)

        #cdist computes n*(n-1)/2 pairwise distances efficiently, "correlation" distance: D_ij = 1 - pearson(x_i, x_j),
        return cdist(X_flat, X_flat, metric=self.dissimilarity)
        

    def compare_rdms(self, rdm1: np.ndarray, rdm2: np.ndarray) -> float:
        """
        Compare two RDMs by correlating their upper triangles.
        """
        ### TODO
        n   = rdm1.shape[0]
        idx = np.triu_indices(n, k=1) #selects elements strictly above the diagonal, no self distances 
        vec1 = rdm1[idx]
        vec2 = rdm2[idx]

        #computes correlation between the 2 upper triangles of the RDMs
        if self.similarity_metric == "spearman":
            r, _ = spearmanr(vec1, vec2)
        else:
            r, _ = pearsonr(vec1, vec2)

        return float(r)

## 2.2 Representational alignment: unbiased linear CKA

CKA stands for centered kernel alignment. It is commonly used in interpretability and representation analysis to test how strongly the internal computations of two systems align. As a second mapping-free alignment metric, we want to compute unbiased linear centered kernel alignment (CKA) between model features and neural responses. Let

$$
X \in \mathbb{R}^{n \times d}, \qquad
Y \in \mathbb{R}^{n \times p},
$$

where both matrices are measured on the same $n$ stimuli. We form linear Gram matrices

$$
K = XX^\top,
\qquad
L = YY^\top.
$$

We then estimate dependence using the unbiased (U-statistic) HSIC estimator, $\mathrm{HSIC}_u(K, L)$, and define CKA as

$$
\mathrm{CKA}(X, Y)
=
\frac{\mathrm{HSIC}_u(K, L)}
{\sqrt{\mathrm{HSIC}_u(K, K)\,\mathrm{HSIC}_u(L, L)}}.
$$

Like RSA, CKA compares representational structure directly without fitting a predictive mapping. In this notebook, `X` and `Y` again refer to aligned model and neural response matrices evaluated on the same set of stimuli.

- Implement **unbiased linear CKA** only.
- Use your implementation to compare model layers to neural responses.

### Starter code

In [ ]:
import numpy as np

class CenteredKernelAlignment:
    """
    Unbiased linear CKA only.

    Parameters
    ----------
    eps : float
        Small constant for numerical stability.
    dtype : np.dtype
        Data type used for computations.
    """

    def __init__(
        self,
        eps: float = 1e-8,
        dtype: np.dtype = np.float64,
    ):
        ### TODO
        pass

    def __call__(self, X: np.ndarray, Y: np.ndarray) -> float:
        return self.forward(X, Y)

    def forward(self, X: np.ndarray, Y: np.ndarray) -> float:
        X = np.asarray(X).astype(self.dtype)
        Y = np.asarray(Y).astype(self.dtype)

        if X.shape[0] != Y.shape[0]:
            raise ValueError(
                f"Batch sizes must match along axis 0: {X.shape[0]} vs {Y.shape[0]}"
            )

        # Flatten to (n_samples, n_features)
        X = X.reshape(X.shape[0], -1)
        Y = Y.reshape(Y.shape[0], -1)

        return self._unbiased_linear_cka(X, Y)

    def _unbiased_linear_hsic(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Unbiased HSIC estimator for the linear kernel.

        X : [n, d_x]
        Y : [n, d_y]
        """
        ### TODO
        pass

    def _unbiased_linear_cka(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Unbiased linear CKA:

            CKA_unb(X, Y) =
                HSIC_unb(X, Y) / sqrt(HSIC_unb(X, X) * HSIC_unb(Y, Y))
        """
        ### TODO
        pass

## 2.3 Apply RSA and CKA

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

- Compare layers within each model.
- Compare the two models.
- For EEG, show how representational similarity changes over time.
- For TVSD and NSD, compare across ROIs.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **Layer-wise RSA results** for both models.
2. **Layer-wise CKA results** for both models.
3. **One direct comparison between the two models** using representational metrics.
4. **One EEG time-resolved analysis** or **one ROI-wise analysis** for TVSD/NSD.
5. **One short written interpretation** in Answer box 2.1.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.1</strong><br>Do RSA and CKA tell the same story? Identify at least one case where they agree and one case where they disagree, and explain what that might mean.</div>

In [ ]:
# TODO: implement RSA
# TODO: implement unbiased linear CKA
# TODO: compute scores across layers
# TODO: compare across models and ROIs / time windows

---

## 2.4 Predictive alignment: linear encoding models

## Linear encoding model

In the predictive part of the project, you will map model features to neural responses using a **linear encoding model with L2 regularization (ridge regression)**.

For a stimulus $\mathbf{x}$, let $\mathbf{z}_{\ell}(\mathbf{x})$ denote the feature vector extracted from model layer $\ell$. For a given subject $s$ and neural target $r$ (for example an ROI, a group of voxels, or a set of channels / time points), the predicted neural response is

$$
\widehat{\mathbf{y}}_{r,s}(\mathbf{x})
=
W_{r,s}\,\mathbf{z}_{\ell}(\mathbf{x}) + \mathbf{b}_{r,s},
$$

where:
- $\mathbf{z}_{\ell}(\mathbf{x}) \in \mathbb{R}^{d}$ is the model feature vector,
- $\widehat{\mathbf{y}}_{r,s}(\mathbf{x}) \in \mathbb{R}^{p}$ is the predicted neural response,
- $W_{r,s} \in \mathbb{R}^{p \times d}$ is the learned linear mapping,
- $\mathbf{b}_{r,s} \in \mathbb{R}^{p}$ is a bias term.

We fit the mapping on the training split using ridge regression:

$$
\min_{W_{r,s},\,\mathbf{b}_{r,s}}
\sum_{\mathbf{x}\in\mathcal{D}_{\mathrm{train}}}
\left\|
\mathbf{y}_{r,s}(\mathbf{x}) - \widehat{\mathbf{y}}_{r,s}(\mathbf{x})
\right\|_2^2
\;+\;
\alpha \left\|W_{r,s}\right\|_F^2.
$$

Here, $\mathbf{y}_{r,s}(\mathbf{x})$ is the measured neural response, and $\alpha$ controls the strength of L2 regularization. Larger $\alpha$ penalizes large weights more strongly, which can improve generalization when the feature dimension is high. You should select $\alpha$ using only the training data, for example with a validation split or cross-validation, and then evaluate the final model on the held-out test set.

<div style="background:#eef8f4; border-left:4px solid #5b9a7a; padding:8px 12px; border-radius:6px; font-weight:700; color:#285943;">Select the required targets</div>

Use the following targets:

- **TVSD:** all ROIs
- **EEG2:** `occipital_parietal`
- **NSD:** `V1v`, `V2v`, `V3v`, `hV4`, `FFA-1`, `VWFA-1`, `PPA`, `OPA`, `EBA`

You may explore additional ROIs if you wish.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.2</strong><br>Briefly explain why these targets are scientifically interesting. Are they chosen mainly for reliability, interpretability, or both?</div>

In [ ]:
# TODO: define target ROIs / regions
# TODO: load corresponding neural data

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

For each dataset, target, model, and candidate layer:

- fit a **linear encoding model**,
- select hyperparameters without using the test split,
- evaluate on the test split.

Use iterative solvers (e.g. SGD, Adam) when needed to avoid memory issues, since `sklearn` Ridge might cause OOM.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **A clearly defined train/validation/test procedure** that does not use the test set for model selection.
2. **Linear encoding model results** for all required datasets and targets.
3. **The following predictive metrics:** Pearson correlation, noise-corrected Pearson correlation, explained variance, and noise-corrected explained variance.
4. **The following hybrid representational metrics on predicted responses:** encoding-RSA and encoding-CKA.
5. **Layer-wise plots** showing performance across candidate layers.
6. **One best-layer summary table** for the required targets.
7. **One comparison between the two models** using predictive results.
8. **One short written interpretation** in Answer box 2.3.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.3</strong><br>Which model and which layer perform best for each dataset? Summarize the main trends in a short paragraph.</div>

In [ ]:
# TODO: define train/val/test procedure
# TODO: fit linear models across layers
# TODO: compute predictive metrics
# TODO: summarize best layers and best scores

---

## 2.5 Compare predictive and representational metrics

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Compare the ranking of models and layers according to:

- Pearson correlation,
- explained variance,
- RSA,
- CKA.
- encoding-RSA/ encoding-CKA

Discuss whether the same layers are favored by all metrics.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One figure comparing layer or model rankings across metrics**
2. **One concrete example where two metrics agree**
3. **One concrete example where two metrics disagree**
4. **One short written interpretation** in Answer box 2.4.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.4</strong><br>Does a model that is representationally similar to the brain also predict neural responses well? Use at least one example from your results.</div>

In [ ]:
# TODO: compare ranking of layers across metrics
# TODO: identify agreements and disagreements

---

## 2.6 Relate layer hierarchy to brain hierarchy

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Test whether deeper layers align better with higher-level neural targets.

- Does TVSD IT align with deeper layers than V1?
- Do higher-level NSD regions prefer later layers?
- For EEG, are particular time windows associated with later layers?

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include one of the following analyses:

1. **A heatmap of layer × ROI**
2. **A ranked-layer plot by ROI**
3. **A time-resolved EEG layer comparison**

You must also include a short written conclusion in Answer box 2.5 stating whether the results support a hierarchy correspondence.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.5</strong><br>Is there evidence for a correspondence between model depth and neural hierarchy? State your conclusion clearly and support it with results.</div>

In [ ]:
# TODO: compare layers across ROIs / time windows
# TODO: create hierarchy figure(s)

---

## 2.7 Compare the two feature extractors

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must do</div>

Compare **Qwen3-VL-2B-Instruct** and **Adv-ResNet152** across datasets, ROIs, layers, and metrics.

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **One summary figure comparing Qwen3-VL-2B-Instruct and Adv-ResNet152**
2. **One table of best scores across datasets and targets**
3. **One short written interpretation** in Answer box 2.6.

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 2.6</strong><br>Does the vision-language model provide a clear advantage over the CNN? Is that advantage consistent across modalities and targets?</div>

In [ ]:
# TODO: aggregate results across both models
# TODO: create summary comparison figure/table

---

# 3. Open-Ended Research

So far you have explored a simple encoding model with a linear readout from a single layer per subject/ROI. In this section, you will extend the baseline pipeline in one clearly defined direction. The goal is to explore a meaningful extension that goes beyond the standard linear readout and to evaluate whether it provides a practically meaningful improvement. Depth is more important than breadth: a focused experiment is better than a broad but shallow exploration.

Possible directions include:

- readouts shared across ROIs,
- readouts shared across subjects,
- readouts shared across modalities,
- combining multiple layers,
- low-rank readouts,
- nonlinear readouts,
- temporal readouts for EEG,
- attention-based readouts,
- cross-subject pooling.

## What you must include

1. **Question**  
   What are you testing?

2. **Motivation**  
   Why is this extension interesting?

3. **Method**  
   What did you change relative to the linear baseline?

4. **Comparison**  
   How does it compare to the baseline?

5. **Interpretation**  
   Did it help, and why might that be?

<div style="background:#f3f6fa; border-left:4px solid #7a93ac; padding:8px 12px; border-radius:6px; font-weight:700; color:#32475b;">Required deliverables</div>

You must include all of the following:

1. **A clearly stated hypothesis**
2. **A short motivation for the extension**
3. **A clear description of the new method**
4. **One direct comparison against the linear baseline**
5. **At least one figure or one table summarizing the comparison**
6. **A short discussion of whether the extension helped in a practically meaningful way**

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box 3</strong><br>State your hypothesis, summarize your result, and say whether the new method helped in a practically meaningful way.</div>

In [ ]:
# TODO: define extension
# TODO: implement method
# TODO: compare against linear baseline

---

# Final Discussion

End the notebook with a short final discussion.

<div style="background:#eef5fb; border-left:4px solid #4c78a8; padding:8px 12px; border-radius:6px; font-weight:700; color:#26445e;">What you must address</div>

- Which dataset appeared noisiest?
- Which neural targets were most reliable?
- Which model aligned best overall?
- Which metrics were most consistent with each other?
- What was the main limitation of your analysis?
- What would you try next with more time?

<div style="background:#fff4c2; border:1px solid #c89b1f; border-left:6px solid #9a6f00; padding:10px 12px; border-radius:6px; margin-top:8px; margin-bottom:4px; color:#241a00; line-height:1.45;"><strong style="color:#5c4300;">Answer box final</strong><br>Write a concise final conclusion of 1–2 paragraphs summarizing your main findings and their limitations.</div>

In [ ]:
# No code required here unless you want to add final summary tables/figures.

---

# Report Content

Your **2-page PDF report** should tell a clear and coherent story. It does **not** need to reproduce every notebook result.

It can include:

1. **A brief dataset overview**
2. **An exploratory figure from Section 1**
3. **The EEG noise ceiling comparison**
4. **The NSD reliability visualization**
5. **One or two key brain–model alignment results from Section 2**


Rather, you should primarily focus on the open-ended extension you designed, describing:
- the motivation for your extension,
- the methods you implemented,
- the results you obtained,
- and the scientific insights you gained from it.


The report should emphasize interpretation, not just figures. Since the notebook is the main technical deliverable, the report should act as a **compressed scientific summary** of your most important findings rather than a figure dump.

---

# Detailed Grading Rubric

The project is graded out of **100 points** as follows:

- **Section 1: Inspection, Visualization, and Noise Ceiling Estimates — 20 points**
- **Section 2: Brain–Model Alignment — 20 points**
- **Section 3: Open-Ended Research — 30 points**
- **Report — 30 points**

**Section 0 is required but not graded separately.** It is treated as setup and reproducibility infrastructure for the rest of the notebook.

## Section 1 — 20 points

### 1.1 Dataset inspection — 3 points
- 1 pt: TVSD structure is correctly inspected and explained.
- 1 pt: EEG2 structure is correctly inspected and explained.
- 1 pt: NSD structure is correctly inspected and explained.

### 1.2 EEG visualization — 4 points
- 1 pt: example EEG time-course plot is present and readable.
- 1 pt: channel × time heatmap is present and readable.
- 1 pt: provided EEG noise ceiling visualization is present and readable.
- 1 pt: written interpretation identifies informative time windows or channel groups.

### 1.3 EEG noise ceiling estimation — 7 points
- 2 pts: variance-based estimator is implemented correctly.
- 2 pts: split-half estimator is implemented correctly.
- 1 pt: required summary visualizations are included.
- 1 pt: comparison to stored EEG noise ceilings is shown clearly.
- 1 pt: Answer box 1.3 interprets similarities and differences between estimators.

### 1.4 Statistical comparison of EEG noise ceilings — 3 points
- 1 pt: quantitative comparison table is present.
- 1 pt: statistical test or formal comparison is appropriate and correctly interpreted.
- 1 pt: final conclusion is clearly justified.

### 1.5 NSD reliability visualization — 3 points
- 1 pt: ncsnr is correctly converted and visualized on cortex.
- 1 pt: parcel overlay or parcel-wise summary is included.
- 1 pt: Answer box 1.5 correctly interprets reliable and unreliable regions.

## Section 2 — 20 points

### 2.1 RSA implementation — 3 points
- 1 pt: RDM computation is correct.
- 1 pt: RDM comparison is correct.
- 1 pt: implementation is used properly in later analyses.

### 2.2 Unbiased linear CKA implementation — 3 points
- 2 pts: unbiased linear CKA is implemented correctly.
- 1 pt: implementation is used properly in later analyses.

### 2.3 Representational analyses across layers, models, and targets — 4 points
- 1 pt: layer-wise RSA results are reported clearly.
- 1 pt: layer-wise CKA results are reported clearly.
- 1 pt: a model comparison is included.
- 1 pt: ROI-wise or time-resolved analysis is included and interpreted.

### 2.4 Predictive alignment with linear encoding models — 6 points
- 1 pt: required targets are selected and described correctly.
- 2 pts: train/validation/test procedure and ridge fitting are correct.
- 1 pt: required predictive metrics are reported correctly.
- 1 pt: encoding-RSA and encoding-CKA are reported correctly.
- 1 pt: best-layer summary and model comparison are included.

### 2.5 Compare predictive and representational metrics — 2 points
- 1 pt: ranking comparison figure is present and informative.
- 1 pt: agreement and disagreement between metrics are discussed clearly.

### 2.6 Layer hierarchy vs brain hierarchy — 1 point
- 1 pt: at least one hierarchy analysis is included and interpreted correctly.

### 2.7 Compare the two feature extractors — 1 point
- 1 pt: final comparison between Qwen3-VL and Adv-ResNet is clear and supported by results.

## Section 3 — 30 points

### Research question and motivation — 5 points
- 2 pts: research question is clear and focused.
- 3 pts: motivation is scientifically sensible and well connected to the baseline project.

### Method and implementation — 10 points
- 4 pts: the extension is described clearly.
- 4 pts: the method is implemented correctly.
- 2 pts: the design remains focused and technically appropriate for the project scope.

### Baseline comparison and evaluation — 10 points
- 4 pts: the comparison to the linear baseline is fair.
- 3 pts: at least one figure or table communicates the comparison clearly.
- 3 pts: evaluation supports the stated conclusion.

### Interpretation and limitations — 5 points
- 3 pts: the student explains whether the method helped in a practically meaningful way.
- 2 pts: limitations or caveats are acknowledged.

## Report — 30 points

### Structure and clarity — 6 points
- clear organization, readable flow, and concise scientific writing.

### Selection of results — 6 points
- the report focuses on the strongest and most relevant results rather than trying to include everything.

### Methodological correctness — 6 points
- metrics, comparisons, and claims are described accurately.

### Interpretation and synthesis — 6 points
- the report explains what the results mean and ties them back to the project goals.

### Figure quality and presentation — 6 points
- figures are readable, labeled, well-chosen, and integrated into the narrative.

## Important grading note

A submission that is technically correct but poorly interpreted will lose points. A submission with good intuition but missing required analyses will also lose points. The strongest submissions will be both **correct** and **scientifically well explained**.

---

# Final Checklist Before Submission

Before submitting, make sure that:

- group information is filled in,
- the notebook runs from top to bottom,
- all notebook outputs are cleared,
- figures have readable titles and labels,
- written answers are included in the answer boxes,
- the zip archive name follows the required format,
- no large unnecessary files are included.

---

# References

Use the references below when you need scientific context for the datasets, models, and analysis methods.

## Datasets

- Papale et al. (2025) — *An extensive dataset of spiking activity to reveal the syntax of the ventral stream*
- Gifford et al. (2022) — *A large and rich EEG dataset for modeling human visual object recognition*
- Allen et al. (2022) — *A massive 7T fMRI dataset to bridge cognitive neuroscience and artificial intelligence*
- van Bree, Styrnal, and Hebart (2025) — *How Much Variance Does Your Model Explain? A Clarifying Note On The Use Of Split-Half Reliability For Computing Noise Ceilings*

## Models

- Wong et al. (2020) — *Fast is better than free: Revisiting adversarial training*
- He et al. (2016) — *Deep Residual Learning for Image Recognition*
- Bai et al. (2025) — *Qwen3-VL Technical Report*

## Alignment and encoding

- Conwell et al. (2024) — *A large-scale examination of inductive biases shaping high-level visual representation in brains and machines*
- Gokce and Schrimpf (2025) — *Scaling Laws for Task-Optimized Models of the Primate Visual Ventral Stream*

Use these references selectively. You are not expected to read everything in full.